# 06 — Model Evaluation (OOT)

Evaluate the trained model on the out-of-time test set:
- MAE and MAPE per horizon (1h → 24h)
- Quantile calibration (80% interval coverage)
- Error analysis by hour-of-day, day-of-week, and price regime

**Prerequisites:** Model trained (NB05).

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import duckdb
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from ml.train import FEATURE_COLS, TARGET_COLS
from ml.evaluate import evaluate_oot, evaluate_quantile_models, mean_absolute_error_per_horizon

con = duckdb.connect()
ABT_TEST = '../data/gold/abt_test.parquet'
MODELS_DIR = Path('../models')

## 6.1 OOT metrics

In [ ]:
model = joblib.load(MODELS_DIR / 'lgbm_multioutput.pkl')
metrics = evaluate_oot(model)
q_metrics = evaluate_quantile_models()
metrics.update(q_metrics)

print(f"Mean MAE (all horizons): {metrics['mae_mean_all']:.4f} EUR/MWh")
print(f"Mean MAPE: {metrics.get('mape_mean_all', 0):.2f}%")
if 'interval_coverage_80pct' in metrics:
    print(f"80% interval coverage: {metrics['interval_coverage_80pct']*100:.1f}% (target ~80%)")

Path('../models/metrics.json').write_text(json.dumps(metrics, indent=2))
print('Metrics saved.')

## 6.2 MAE per horizon

In [ ]:
horizons = list(range(1, 25))
mae_vals = [metrics[f'mae_t_plus_{h}h'] for h in horizons]

fig = go.Figure(go.Bar(x=[f't+{h}h' for h in horizons], y=mae_vals,
                       marker_color=mae_vals, marker_colorscale='Blues',
                       text=[f'{v:.2f}' for v in mae_vals], textposition='outside'))
fig.update_layout(title='MAE per Forecast Horizon — OOT Test Set',
                  xaxis_title='Horizon', yaxis_title='MAE (EUR/MWh)',
                  template='plotly_dark', height=400)
fig.show()

## 6.3 Error analysis by hour-of-day

In [ ]:
test_df = con.execute(f'SELECT * FROM read_parquet("{ABT_TEST}")').fetchdf()
preds = model.predict(test_df[FEATURE_COLS])
actuals = test_df[TARGET_COLS].values

# Use t+12h as representative mid-horizon
H = 11
errors = np.abs(actuals[:, H] - preds[:, H])
test_df['abs_error_12h'] = errors
test_df['hour_of_day'] = pd.to_datetime(test_df['timestamp_utc'], utc=True).dt.hour

df_hour_err = test_df.groupby('hour_of_day')['abs_error_12h'].mean().reset_index()

fig = px.bar(df_hour_err, x='hour_of_day', y='abs_error_12h',
             title='Mean |Error| at t+12h by Hour of Day',
             labels={'hour_of_day': 'Hour (UTC)', 'abs_error_12h': 'MAE (EUR/MWh)'},
             template='plotly_dark')
fig.show()

## 6.4 Error by price regime (high/mid/low)

In [ ]:
p33, p67 = np.percentile(test_df['price_eur_mwh'].dropna(), [33, 67])
test_df['price_regime'] = pd.cut(test_df['price_eur_mwh'],
                                   bins=[-999, p33, p67, 999],
                                   labels=['Low', 'Mid', 'High'])

df_regime = test_df.groupby('price_regime')['abs_error_12h'].agg(['mean','std','count'])
print('MAE by price regime (t+12h):')
print(df_regime.round(3))